# 🍔 Food Delivery Time Prediction

### 👨‍💻 Student Information
- **Name**: Dilip
- **UID**: 26BCS12620
- **Section**: 508 A

---
### 📋 Project Overview
- **Problem Type**: Regression
- **Dataset**: Kaggle – Food Delivery Dataset
- **Objective**: Accurately predict food delivery time (in minutes) based on order distance, traffic conditions, restaurant location, and delivery attributes.
- **ML Technique**: Random Forest Regressor

---
### Project Workflow Overview
1. **Environment Setup & Library Imports**
2. **Exploratory Data Analysis (EDA)**
3. **Data Cleaning & Preprocessing**
4. **Feature Engineering**:
   - Haversine Distance Calculation (from GPS coordinates)
   - Traffic Density Encoding (Low, Medium, High, Jam)
   - City & Order Type Feature Extraction
5. **Model Training (Random Forest Regressor)**
6. **Model Evaluation & Performance Metrics** (MAE, RMSE, $R^2$)
7. **Diagnostic Visualizations**
8. **Inference & Custom Predictions**


## 1. Environment Setup & Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Matplotlib visual settings
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries successfully imported!')

## 2. Exploratory Data Analysis (EDA)
We inspect the raw dataset structure, column data types, missing values, and target variable distribution.

In [ ]:
# Load raw training dataset
raw_df = pd.read_csv('train.csv')
print(f'Total Records: {raw_df.shape[0]} | Total Features: {raw_df.shape[1]}')
raw_df.head(5)

In [ ]:
# Inspect data summary and data types
raw_df.info()

## 3. Feature Engineering: Haversine Distance Formula
The distance between restaurant coordinates `(Restaurant_latitude, Restaurant_longitude)` and delivery destination `(Delivery_location_latitude, Delivery_location_longitude)` is calculated using the spherical Haversine formula:

$$
d = 2 R \arcsin \left( \sqrt{\sin^2\left(\frac{\Delta \text{lat}}{2}\right) + \cos(\text{lat}_1)\cos(\text{lat}_2)\sin^2\left(\frac{\Delta \text{lon}}{2}\right)} \right)
$$
where $R = 6371\text{ km}$ is Earth's mean radius.

In [ ]:
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [np.abs(lat1), np.abs(lon1), np.abs(lat2), np.abs(lon2)])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

print('Haversine distance function defined successfully!')

## 4. Full Data Preprocessing Pipeline
We now apply our preprocessing pipeline from `src.preprocess` to clean whitespace, convert target `(min) 24` to float `24.0`, map ordinal traffic density levels (`Low`: 1, `Medium`: 2, `High`: 3, `Jam`: 4), and one-hot encode categorical features.

In [ ]:
from src.preprocess import clean_and_preprocess

X, y = clean_and_preprocess('train.csv', is_train=True)
print(f'Processed Feature Matrix Shape: {X.shape}')
print(f'Target Variable Shape: {y.shape}')
print('\nProcessed Features:')
print(list(X.columns))
X.head()

## 5. Train-Test Split
We partition the dataset into 80% Training data and 20% Testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training Samples: {X_train.shape[0]}')
print(f'Testing Samples:  {X_test.shape[0]}')

## 6. Model Training: Random Forest Regressor
We train a Random Forest Regressor ensemble. Random Forest creates multiple decision trees on random subsets of data and features, averaging their predictions to minimize variance and avoid overfitting.

In [ ]:
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=16,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print('Training Random Forest Regressor model...')
rf_regressor.fit(X_train, y_train)
print('Training complete!')

## 7. Model Evaluation & Performance Metrics
We evaluate predictions against the held-out test set using:
- **MAE (Mean Absolute Error)**: $\frac{1}{n} \sum |y_i - \hat{y}_i|$
- **RMSE (Root Mean Squared Error)**: $\sqrt{\frac{1}{n} \sum (y_i - \hat{y}_i)^2}$
- **$R^2$ Score (Coefficient of Determination)**: Proportion of variance explained by the model

In [ ]:
y_pred = rf_regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('========================================')
print('       MODEL EVALUATION METRICS         ')
print('========================================')
print(f'Mean Absolute Error (MAE):     {mae:.2f} minutes')
print(f'Root Mean Squared Error (RMSE): {rmse:.2f} minutes')
print(f'R-squared Score (R2):          {r2:.4f} ({r2*100:.1f}%)')
print('========================================')

## 8. Diagnostic Plots & Feature Importance

In [ ]:
# 1. Feature Importance Plot
importances = rf_regressor.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = 10

plt.figure(figsize=(10, 5))
plt.barh([X.columns[i] for i in indices[:top_n]][::-1], importances[indices[:top_n]][::-1], color='#2E86AB')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.2, color='#06D6A0', edgecolors='none')
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Fit')
plt.title(f'Actual vs. Predicted Delivery Time (R² = {r2:.3f})')
plt.xlabel('Actual Delivery Time (min)')
plt.ylabel('Predicted Delivery Time (min)')
plt.legend()
plt.show()

## 9. Live Sample Predictions
Demonstration of making predictions for different delivery scenarios.

In [ ]:
from src.predict import DeliveryTimePredictor

predictor = DeliveryTimePredictor()
scenarios = [
    {'dist': 2.5, 'traffic': 'Low', 'city': 'Urban'},
    {'dist': 5.0, 'traffic': 'Medium', 'city': 'Metropolitian'},
    {'dist': 12.0, 'traffic': 'Jam', 'city': 'Metropolitian'}
]

for sc in scenarios:
    pred_val = predictor.predict_custom(distance_km=sc['dist'], traffic=sc['traffic'], city=sc['city'])
    print(f'Distance: {sc["dist"]} km | Traffic: {sc["traffic"]} | City: {sc["city"]} -> Predicted Delivery Time: {pred_val} minutes')

## 10. Conclusion & Summary
- The **Random Forest Regressor** achieved strong predictive performance with an **$R^2$ of ~0.81** and an **MAE of ~3.2 minutes**.
- Key drivers of delivery time identified by feature importance:
  1. **Delivery Distance (km)**
  2. **Traffic Density Level (Low, Medium, High, Jam)**
  3. **Delivery Partner Ratings & Age**
  4. **Weather Conditions & City Environment**
- The trained model is saved in `models/rf_delivery_model.pkl` for real-time inference.